# 08 · Escenario común del equipo: 60 GBd a 1,2 sps (D1.1)

Verificación con el golden en flotante del escenario del documento común del equipo (21/11/2026): DP-16QAM, 60 GBd, 1,2 muestras por símbolo (6/5), f<sub>s</sub> = 72 GSa/s, 4 canales reales XI, XQ, YI, YQ en 16 bits Q1.15 (`ac_fixed<16, 1>`).

**Qué se mide**
- **Parte 1 · Rango en Q1.15:** RMS, pico, factor de cresta, muestras con |v| ≥ 1 y SQNR por cuantizar a Q1.15, para la señal común y para las salidas de la CD (0,1 a 120 km) y de la ISI (7 perfiles × 4 anchos de banda, con dos normalizaciones de h). Cota de peor caso del acumulador de la ISI.
- **Parte 2 · CD a distancias cortas:** FIR complejo por mínimos cuadrados (T2), forma de primer orden y extendida (T3), precisión reducida de la corrección a 100 m (T4).
- **Parte 3 · ISI a 1,2 sps:** menor K por cuatro criterios (energía, truncado, mínimos cuadrados en frecuencia y sobre la señal) contra el prototipo sin truncar (T5, T6).
- **Parte 4 · DSP por fórmula** (estimación, no síntesis) para la ISI con FFA y la CD de primer orden (T7).
- **D1.2:** criterio (e) de la ISI, mínimos cuadrados en frecuencia pesados por |H<sub>RRC</sub>(f)|² (T5, T6); pico y bits enteros del acumulador de la ISI (T5b); CD entre 10 y 20 km en Q1.15 (T1b).

**Qué NO se mide**
- No hay punto fijo del hardware ni RTL: T4 es un modelo de precisión reducida con productos y sumas exactos, no la arquitectura.
- No hay receptor ni EVM: todo se compara contra una referencia en flotante (NRMSE o SQNR).
- No se usa la arquitectura de la CD por overlap-save ni sus anchos (decisiones 30 a 32): la referencia es `cd_reference` (FFT de toda la señal).
- No se toman decisiones: al final, "Para decidir" junta los datos.

**Convenciones**
- Funciones del golden sin modificar: `cd_reference`, `cd_R`, `gaussian`, `analog` (Bessel con `norm="mag"`), `K_rule`, `nrmse`, `quantize` (redondeo a par en empates, saturación; decisión 26). Escenario con `Scenario(Rs=60e9, sps=Fraction(6, 5))` (B5.4).
- Funciones auxiliares en `notebooks/utils_08.py`. Resultados en `docs/golden_model/resultados/08_escenario_60gbd/` (`fig/` PNG y SVG, `csv/`, `meta.json`).

In [ ]:
import time
from datetime import datetime, timezone

import matplotlib.pyplot as plt
import numpy as np
import scipy
from IPython.display import Markdown, display

import utils_08 as u
from emulador import plots
from emulador.isi.profiles import K_rule
from emulador.metrics import group_delay
from emulador.quantization import quantize

plots.use_style()
T0 = time.perf_counter()
TIMES = {}
FIG = {"blue": "#2a78d6", "orange": "#eb6834", "aqua": "#1baf7a", "muted": "#7A8499", "ink": "#14213D"}
FAMILY = {"gaussian": FIG["blue"], "bessel": FIG["orange"], "butter": FIG["aqua"]}
MARK = {None: "o", 2: "s", 4: "^", 6: "D"}
SEQ8 = ["#86b6ef", "#6da7ec", "#5598e7", "#3987e5", "#2a78d6", "#256abf", "#184f95", "#0d366b"]
show = lambda rows, cols=None, d=4: display(Markdown(u.md_table(rows, cols, d)))
print(u.ESC, "fs =", u.FS / 1e9, "GSa/s")

## Señal común

4 PRBS15 (ITU-T O.150, x<sup>15</sup> + x<sup>14</sup> + 1, sin inversión; decisión 16), una por tributario. Los estados iniciales son los de la PRBS con semilla 1 desplazada 0, ⌊P/4⌋, ⌊P/2⌋ y ⌊3P/4⌋ bits (P = 32 767), asignados a XI, XQ, YI, YQ en ese orden. 2 bits por símbolo, MSB primero, Gray 00→−3, 01→−1, 11→+1, 10→+3, escalado por 1/3. Conformado con RRC de roll-off 0,1 y span de 32 símbolos a 6 muestras por símbolo, diezmado por 5. Un factor común a los 4 canales lleva el RMS medio a 0,25.

In [ ]:
t = time.perf_counter()
SIG = u.common_signal()
CH = SIG["channels"]
X, Y = CH[0] + 1j * CH[1], CH[2] + 1j * CH[3]
N = CH.shape[1]
rows = [{"canal": c, "offset_bits": o, "semilla": s, "RMS_sin_escala": r0, "RMS": r, "pico": float(np.max(np.abs(v)))}
        for c, o, s, r0, r, v in zip(u.CHANNELS, u.OFFSETS, SIG["seeds"], SIG["rms_raw"], SIG["rms"], CH)]
show(rows)
print(f"muestras por canal: {N}  (81 920 × 6/5 = {u.N_SYM * 6 // 5});  factor común: {SIG['scale']:.6f}")
u.save_csv(rows, "senal_comun_canales")

In [ ]:
xc = []
for i in range(4):
    for j in range(i + 1, 4):
        r1, l1 = u.xcorr_max(SIG["levels"][i], SIG["levels"][j], 1024)
        r2, l2 = u.xcorr_max(SIG["levels"][i], SIG["levels"][j], u.N_SYM // 2)
        xc.append({"par": f"{u.CHANNELS[i]}-{u.CHANNELS[j]}", "max|rho| (|lag|<=1024 símb.)": r1, "lag": l1,
                   "max|rho| (|lag|<=40960 símb.)": r2, "lag ": l2})
show(xc, d=3)
u.save_csv(xc, "senal_comun_correlacion")
TIMES["senal_comun_s"] = time.perf_counter() - t

**Observaciones.**

- Con el factor común 0,821171 el RMS por canal queda entre 0,2499 y 0,2501 (sin escalar: 0,3044 a 0,3045) y el pico de entrada es 0,6951 (XI, YI) y 0,6979 (XQ, YQ). Cada canal tiene 98 304 muestras.
- Para |lag| ≤ 1024 símbolos, la correlación cruzada entre tributarios es |ρ| ≤ 0,0033 en los 6 pares.
- **Pero no son independientes a desfasajes largos:** |ρ| = 1 a ±12 288, ±36 863 y ±40 959 símbolos. Las cuatro son la misma secuencia m desplazada; con 2 bits por símbolo y período impar (32 767), la secuencia de símbolos de un tributario es un desplazamiento circular de la de otro. Esos desfasajes superan la memoria de todos los filtros del notebook: el prototipo de la ISI tiene 8192 muestras, unos 6827 símbolos, y el chirp de la CD a 120 km, R = 85 muestras.
- **Pendiente con el equipo:** con un solo polinomio PRBS15, cualquier par de estados iniciales son desplazamientos de la misma secuencia m. Las opciones son un polinomio primitivo distinto por tributario o una PRBS más larga (GUIA §6). Para esta tarea queda así.

## Parte 1 · Rango en Q1.15 (¿alcanza 1 bit entero?)

Para cada canal real: RMS, pico, factor de cresta (pico/RMS), muestras con |v| ≥ 1, muestras que modifica la saturación y dos SQNR: **completa** (Q1.15 con redondeo y saturación contra la señal en flotante) y **solo saturación** (Q1.15 con saturación contra Q1.15 sin saturar; ∞ si no satura). Las tablas muestran el peor caso de los 4 canales (pico y cresta máximos, SQNR mínima, saturaciones sumadas). Se descartan 1024 muestras por borde.

- **CD:** `cd_reference` sobre X e Y, D = 17 ps/nm/km, λ = 1550 nm.
- **ISI:** el prototipo sin truncar de la Parte 3 (8192 coeficientes), aplicado a los 4 canales, con dos normalizaciones de h: (a) la del golden, Σ|h|² = 1; (b) ganancia RMS unitaria sobre la señal común (un factor g por perfil y B, con la potencia media de los 4 canales).
- **Cota del acumulador de la ISI:** Σ|h| × pico de entrada, para el h largo y para la mejor ventana de 64 coeficientes (renormalizada como en (a) o (b)).

In [ ]:
t = time.perf_counter()
E1 = 1024
inner = slice(E1, N - E1)
t1_in = [dict(canal=c, **u.range_stats(v[inner])) for c, v in zip(u.CHANNELS, CH)]
show(t1_in)
PEAK_IN = max(r["peak"] for r in t1_in)
u.save_csv(t1_in, "T1_entrada")

In [ ]:
Z_T1 = (0.1, 2, 5, 8, 10, 20, 80, 120)
t1_cd, t1_cd_ch, CD_OUT = [], [], {}
for S in u.S_PS:
    for z in Z_T1:
        yx, yy = u.cd_apply(X, z, S), u.cd_apply(Y, z, S)
        chans = [yx.real, yx.imag, yy.real, yy.imag]
        CD_OUT[(z, S)] = chans
        st = [u.range_stats(v[inner]) for v in chans]
        t1_cd_ch += [dict(z_km=z, S=S, canal=c, **r) for c, r in zip(u.CHANNELS, st)]
        t1_cd.append(dict(z_km=z, S=S, R=u.cd_R_samples(z), **u.worst_case(st)))
show(t1_cd)
u.save_csv(t1_cd, "T1_cd_peor_caso"); u.save_csv(t1_cd_ch, "T1_cd_por_canal")

In [ ]:
t1_isi, t1_isi_ch, ISI_OUT = [], [], {}
p_in = np.mean(CH[:, inner] ** 2)
for kind, m in u.PROFILES:
    for B in u.B_RELS:
        h = u.h_reference(kind, m, B)
        c0 = int(np.argmax(np.abs(h)))
        ya = np.array([scipy.signal.fftconvolve(v, h)[c0 : c0 + N] for v in CH])
        g = float(np.sqrt(p_in / np.mean(ya[:, inner] ** 2)))
        o64, _ = u.best_window(h, 64)
        h64 = h[o64 : o64 + 64] / np.linalg.norm(h[o64 : o64 + 64])
        for norm, gain in (("a", 1.0), ("b", g)):
            st = [u.range_stats(gain * v[inner]) for v in ya]
            name = u.profile_name(kind, m)
            t1_isi_ch += [dict(perfil=name, B_Rs=B, norm=norm, canal=c, **r) for c, r in zip(u.CHANNELS, st)]
            t1_isi.append(dict(perfil=name, B_Rs=B, norm=norm, g=gain, **u.worst_case(st),
                               cota_largo=gain * np.sum(np.abs(h)) * PEAK_IN,
                               cota_64=gain * np.sum(np.abs(h64)) * PEAK_IN))
            ISI_OUT[(kind, m, B, norm)] = gain * ya
cols = ["perfil", "B_Rs", "norm", "g", "rms_mean", "peak_max", "crest_max", "n_ge1_total", "n_sat_total",
        "sqnr_full_min_db", "sqnr_sat_min_db", "cota_largo", "cota_64"]
show(t1_isi, cols, 3)
u.save_csv(t1_isi, "T1_isi_peor_caso"); u.save_csv(t1_isi_ch, "T1_isi_por_canal")

In [ ]:
def summary(rows, label):
    pk = max(r["peak_max"] for r in rows)
    ns = sum(r["n_sat_total"] for r in rows)
    sq = min(r["sqnr_full_min_db"] for r in rows)
    return {"bloque": label, "pico máx.": pk, "casos con saturación": sum(r["n_sat_total"] > 0 for r in rows),
            "casos": len(rows), "muestras saturadas (total)": ns, "SQNR completa mín. [dB]": sq}

cd_le8 = [r for r in t1_cd if r["z_km"] <= 8]
resumen_t1 = [summary([{"peak_max": PEAK_IN, "n_sat_total": sum(r["n_sat"] for r in t1_in),
                        "sqnr_full_min_db": min(r["sqnr_full_db"] for r in t1_in)}], "entrada"),
              summary(cd_le8, "CD z <= 8 km"), summary(t1_cd, "CD todas"),
              summary([r for r in t1_isi if r["norm"] == "a"], "ISI (a) Σ|h|²=1"),
              summary([r for r in t1_isi if r["norm"] == "b"], "ISI (b) RMS unitario")]
show(resumen_t1)
sat_cd = [(r["z_km"], r["S"], r["n_sat_total"]) for r in t1_cd if r["n_sat_total"]]
print("CD con saturación (z, S, muestras):", sat_cd)
u.save_csv(resumen_t1, "T1_resumen")
TIMES["parte1_s"] = time.perf_counter() - t

**Observaciones.**

- **Entrada:** pico 0,698, cresta 2,78 a 2,79 y ninguna muestra con |v| ≥ 1. La SQNR de Q1.15 es 89,05 dB en los 4 canales; es puro redondeo, porque no hay saturación.
- **CD:** el RMS no cambia (0,2499) y el pico crece con z:
  - 0,699 a 0,1 km, 0,782 a 2 km, 0,894 a 5 km, 0,936 a 8 km y 0,9965 a 10 km, todos sin saturación;
  - a 20 km el pico es 1,15 y satura: 17 muestras de 385 024 (4 canales), SQNR 52,4 dB;
  - a 80 km, 15 muestras (49,9 dB); a 120 km, pico 1,40 y 24 muestras (41,6 dB).
  
  S = 0,057 no cambia ninguno de estos valores en 3 cifras.
- **ISI:**
  - con (a), Σ|h|² = 1, el RMS de salida va de 0,256 a 0,274 (ganancia en potencia > 1) y el pico máximo es 0,744 (Butterworth m = 6, B = 0,5·R<sub>s</sub>);
  - con (b), g entre 0,913 y 0,978, el pico máximo es 0,699;
  - ninguno de los 56 casos satura (SQNR ≥ 89,0 dB);
  - **cota del acumulador** Σ|h|·pico: hasta 3,60 con el h largo (Butterworth m = 2, B = 0,6, norma (a)) y hasta 2,11 con la mejor ventana de 64. Para el gaussiano, entre 0,82 y 1,07.
- **1 bit entero (Q1.15):** alcanza, sin muestras saturadas, para la entrada, la CD hasta 10 km (pico 0,9965, a 0,35 % del fondo de escala) y todas las salidas de la ISI con las dos normalizaciones. No alcanza para la CD desde 20 km (pico 1,15 a 1,40). Entre 10 y 20 km: ver T1b (D1.2).

### T1b · CD entre 10 y 20 km (D1.2)

Cierra el hueco de T1 entre 10 y 20 km con las mismas mediciones: pico, muestras con |v| ≥ 1, saturaciones y SQNR en Q1.15, peor caso de los 4 canales.

In [ ]:
Z_T1B = (10, 12, 14, 16, 18, 20)
t1b = []
for S in u.S_PS:
    for z in Z_T1B:
        yx, yy = u.cd_apply(X, z, S), u.cd_apply(Y, z, S)
        st = [u.range_stats(v[inner]) for v in (yx.real, yx.imag, yy.real, yy.imag)]
        t1b.append(dict(z_km=z, S=S, R=u.cd_R_samples(z), **u.worst_case(st)))
show(t1b)
u.save_csv(t1b, "T1b_cd_10a20km")

**Observaciones.**

- 10 km es la última distancia sin saturación (pico 0,9965). Desde 12 km satura:

  | z | pico | muestras saturadas | SQNR |
  |---|---|---|---|
  | 12 km | 1,079 | 2 | 57,6 dB |
  | 14 km | 1,076 | 3 | 59,3 dB |
  | 16 km | 1,099 | 4 | 55,4 dB |
  | 18 km | 1,108 | 8 | 54,0 dB |
  | 20 km | 1,150 | 17 | 52,4 dB |

  Son muestras de 385 024.
- El umbral de 1 bit entero con esta señal queda entre 10 y 12 km. Con S = 0,057 los resultados son iguales (≤ 0,01 dB de diferencia).
- El pico no crece de forma estrictamente monótona (1,079 a 12 km y 1,076 a 14 km): depende de las pocas muestras extremas de esta señal.

In [ ]:
worst_b = max((r for r in t1_isi if r["norm"] == "b"), key=lambda r: r["peak_max"])
kb = next((k, m) for k, m in u.PROFILES if u.profile_name(k, m) == worst_b["perfil"])
panels = [("Entrada (4 canales)", CH),
          ("CD 8 km, S = 0", np.array(CD_OUT[(8, 0.0)])),
          ("CD 120 km, S = 0", np.array(CD_OUT[(120, 0.0)])),
          (f"ISI (b), mayor pico: {worst_b['perfil']}, B = {worst_b['B_Rs']}·Rs", ISI_OUT[(*kb, worst_b["B_Rs"], "b")])]
fig, axs = plt.subplots(2, 2, figsize=(11, 6.5), sharex=True)
bins = np.linspace(-1.25, 1.25, 251)
for ax, (title, v) in zip(axs.flat, panels):
    vv = v[:, inner].ravel()
    ax.hist(vv, bins=bins, color=FIG["blue"], alpha=0.85)
    ax.set_yscale("log")
    for xl in (-1, u.FS_MAX):
        ax.axvline(xl, color=FIG["muted"], ls="--", lw=1)
    ax.text(1.02, 0.95, "fondo de escala", transform=ax.get_xaxis_transform(), color=FIG["muted"], fontsize=9, va="top")
    ax.set_title(title, fontsize=11)
    ax.text(0.02, 0.95, f"pico {np.max(np.abs(vv)):.3f}", transform=ax.transAxes, fontsize=10, va="top")
for ax in axs[1]:
    ax.set_xlabel("amplitud (fondo de escala = 1)")
for ax in axs[:, 0]:
    ax.set_ylabel("muestras")
fig.suptitle("G1 · Histogramas de entrada y salidas con el fondo de escala de Q1.15", fontweight="bold")
u.save_fig(fig, "G1_histogramas"); plt.show()

**Observaciones.**

- La entrada conserva la forma de 4 niveles del PAM4 filtrado y termina en ±0,70. La salida de la ISI con mayor pico (Butterworth m = 6, B = 0,5, norma (b)) llega a 0,699.
- Con CD la distribución se vuelve más gaussiana y aparecen colas: 0,936 a 8 km y 1,400 a 120 km, con 24 muestras fuera del fondo de escala.

## Parte 2 · CD a distancias cortas

Referencia: `cd_reference` sobre X, con la entrada sin cuantizar. Los ajustes por mínimos cuadrados se hacen sobre la primera mitad de la señal y el NRMSE se mide sobre la segunda, descartando 256 muestras por borde de cada mitad.

- **T2:** FIR complejo centrado de K coeficientes (K impar de 1 a 41). Simetría: ‖h − h invertido‖/‖h‖ (0 si h[k] = h[−k]).
- **T3:** primer orden y = x + j·(s * x), con s real y simétrico de K<sub>s</sub> coeficientes; extendida y = x + (c * x) + j·(s * x), con c real y simétrico de K<sub>c</sub> = 1 o 3. Coeficientes reales por mínimos cuadrados (partes real e imaginaria apiladas).

In [ ]:
t = time.perf_counter()
TR, TE = u.split_indices(N, 256)
Z_T2 = (0.1, 0.2, 0.3, 0.5, 1, 2, 5, 10)
T2 = {}
t2, t2_curve = [], []
for S in u.S_PS:
    for z in Z_T2:
        y = u.cd_apply(X, z, S)
        r = u.fit_fir_complex(X, y, 41, TR, TE)
        T2[(z, S)] = (y, r)
        cur = {K: v[1] for K, v in r.items()}
        t2_curve += [{"z_km": z, "S": S, "K": K, "nrmse": e} for K, e in cur.items()]
        k1, k2 = u.min_K(cur, lambda e: e <= 1e-3), u.min_K(cur, lambda e: e <= 3e-4)
        t2.append({"z_km": z, "S": S, "R": u.cd_R_samples(z), "K(1e-3)": k1 or ">41", "K(3e-4)": k2 or ">41",
                   "NRMSE K=5": cur[5], "NRMSE K=7": cur[7], "NRMSE K=9": cur[9],
                   "asim. K=9": u.asymmetry(r[9][0]), "asim. K=41": u.asymmetry(r[41][0])})
show(t2, d=3)
u.save_csv(t2, "T2_cd_fir_resumen"); u.save_csv(t2_curve, "T2_cd_fir_nrmse_vs_K")

In [ ]:
h7 = T2[(0.1, 0.0)][1][7][0]
print("h (K = 7, z = 0,1 km, S = 0):")
for k, c in enumerate(h7):
    print(f"  h[{k - 3:+d}] = {c.real:+.6f} {c.imag:+.6f}j")

**Observaciones.**

- K mínimo con FIR complejo:

  | z | NRMSE ≤ 1e-3 | NRMSE ≤ 3e-4 |
  |---|---|---|
  | 0,1 km | 5 (7,29e-4) | 9 |
  | 0,2 km | 7 | 11 |
  | 0,3 km | 7 | 11 |
  | 0,5 km | 9 | 13 |
  | 1 km | 11 | 17 |
  | 2 km | 15 | 21 |
  | 5 km | 19 | 31 |
  | 10 km | 25 | > 41 |

  A 0,1 km, K = 7 da 3,14e-4: no llega a 3e-4.
- Con S = 0, h es simétrico: asimetría ≤ 3,1e-4 (10 km, K = 9) y 1,5e-6 a 0,1 km. Con S = 0,057 la asimetría es unas 10 veces mayor (1,3e-5 a 0,1 km y 1,2e-3 a 10 km, K = 9), pero el NRMSE no cambia en 3 cifras.
- A 0,1 km, h (K = 7) tiene el centro en 0,9997 + 0,0177j y los demás coeficientes casi imaginarios puros (parte real ≤ 2,1e-4 y parte imaginaria hasta 0,0105). Es la estructura que aprovecha la forma de primer orden.

In [ ]:
Z_T3 = (0.1, 0.2, 0.3, 0.5)
t3, FO = [], {}
for S in u.S_PS:
    for z in Z_T3:
        y = T2[(z, S)][0]
        for Ks in (3, 5, 7, 9):
            row = {"z_km": z, "S": S, "K_s": Ks}
            for Kc, lab in ((0, "primer orden"), (1, "ext. K_c=1"), (3, "ext. K_c=3")):
                s, c, e = u.fit_first_order(X, y, Ks, Kc, TR, TE)
                row[lab] = e
                FO[(z, S, Ks, Kc)] = (s, c, e)
            row["FIR K=K_s (T2)"] = T2[(z, S)][1][Ks][1]
            t3.append(row)
show(t3, d=3)
u.save_csv(t3, "T3_cd_primer_orden")

**Observaciones.**

- **A 0,1 km, primer orden:** K<sub>s</sub> = 5 da 7,71e-4, 7 da 4,05e-4 y 9 da 2,99e-4. El FIR complejo con el mismo K da 7,29e-4, 3,14e-4 y 1,55e-4. Con K<sub>s</sub> = 5 cumple 1e-3; 3e-4 se cumple recién con K<sub>s</sub> = 9, justo en el límite (2,99e-4).
- **El primer orden se estanca al crecer z:**
  - a 0,2 km el mínimo es 1,07e-3 con K<sub>s</sub> = 9 (no llega a 1e-3); a 0,5 km, 6,43e-3;
  - la extendida con K<sub>c</sub> = 3 baja esos valores a 4,94e-4 (0,2 km) y 2,52e-3 (0,5 km), sin alcanzar al FIR (3,10e-4 y 7,76e-4).
- S = 0,057 da los mismos NRMSE en 3 cifras.

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(15, 4.6), sharey=True)
for ax, S in zip(axs[:2], u.S_PS):
    for col, z in zip(SEQ8, Z_T2):
        cur = T2[(z, S)][1]
        ax.semilogy(list(cur), [v[1] for v in cur.values()], color=col, marker="o", ms=3, label=f"{z:g} km")
    ax.set_title(f"T2 · FIR complejo, S = {S:g} ps/nm²/km", fontsize=11)
    ax.set_xlabel("K (coeficientes)")
ax = axs[2]
for col, z in zip(SEQ8[::2], Z_T3):
    for Kc, ls, mk in ((0, "-", "o"), (3, "--", "s")):
        ks = (3, 5, 7, 9)
        ax.semilogy(ks, [FO[(z, 0.0, k, Kc)][2] for k in ks], color=col, ls=ls, marker=mk, ms=5)
    ax.text(9.15, FO[(z, 0.0, 9, 0)][2], f"{z:g} km", color=FIG["ink"], fontsize=9, va="center")
ax.plot([], [], color=FIG["ink"], ls="-", marker="o", label="primer orden")
ax.plot([], [], color=FIG["ink"], ls="--", marker="s", label="extendida K_c = 3")
ax.set_title("T3 · forma de primer orden, S = 0", fontsize=11); ax.set_xlabel("K_s"); ax.set_xlim(2.5, 10)
for ax in axs:
    for tol in (1e-3, 3e-4):
        ax.axhline(tol, color=FIG["muted"], ls=":", lw=1.2)
        ax.text(ax.get_xlim()[0], tol, f" {tol:g}", color=FIG["muted"], fontsize=9, va="bottom")
axs[0].set_ylabel("NRMSE contra cd_reference")
axs[0].legend(title="z", fontsize=9, ncol=2); axs[2].legend(fontsize=9, loc="lower left")
fig.suptitle("G2 · NRMSE de la CD contra la cantidad de coeficientes", fontweight="bold")
u.save_fig(fig, "G2_cd_nrmse_vs_K"); plt.show()

**Observaciones.**

- Entre K = 1 y K = 9, el NRMSE del FIR baja 1,9 décadas a 0,1 km (1,16e-2 → 1,55e-4) y 1,4 décadas a 10 km (0,873 → 0,0342). Con S = 0 y con S = 0,057 los dos paneles coinciden.
- La forma de primer orden (panel derecho) se aplana por encima de 1e-3 desde 0,2 km. La extendida con K<sub>c</sub> = 3 sigue bajando un poco más.

### T4 · Precisión reducida de la corrección a z = 0,1 km (S = 0)

- **Golden en flotante:** el filtro con coeficientes en flotante (los de T2 y T3) aplicado a la entrada ya cuantizada a Q1.15 (X<sub>q</sub>).
- **Punto fijo:** X<sub>q</sub> pasa directa y solo la corrección usa x redondeado a b<sub>x</sub> bits (`ac_fixed<bx, 1>`) y coeficientes redondeados a b<sub>h</sub> bits con escala potencia de 2 ajustada al máximo |coeficiente| (`ac_fixed<bh, e+1>` con max|c| < 2<sup>e</sup>; misma escala para parte real e imaginaria). Corrección del FIR completo: h − δ (δ en el coeficiente central); de la forma de primer orden: s. Productos y sumas exactos; salida redondeada a Q1.15 con saturación.
- SQNR = 10·log<sub>10</sub>(Σ|y<sub>fl</sub>|² / Σ|y<sub>fx</sub> − y<sub>fl</sub>|²) sobre X complejo, sin 256 muestras por borde.

In [ ]:
XQ = quantize(X, *u.Q115)
SL = slice(256, N - 256)
BX, BH = (6, 8, 10, 12), (4, 5, 6, 8)
cfgs = {"FIR K=7": ("fir", T2[(0.1, 0.0)][1][7][0]), "FIR K=9": ("fir", T2[(0.1, 0.0)][1][9][0]),
        "primer orden K_s=5": ("fo", FO[(0.1, 0.0, 5, 0)][0]), "primer orden K_s=7": ("fo", FO[(0.1, 0.0, 7, 0)][0])}
T4 = {}
t4 = []
for name, (kind, c) in cfgs.items():
    f = u.t4_fir if kind == "fir" else u.t4_first_order
    for bx in BX:
        row = {"config": name, "bx": bx}
        for bh in BH:
            T4[(name, bx, bh)] = f(XQ, c, bx, bh, SL)
            row[f"bh={bh}"] = T4[(name, bx, bh)]
        t4.append(row)
show(t4, d=3)
u.save_csv(t4, "T4_sqnr_precision_reducida")
TIMES["parte2_s"] = time.perf_counter() - t

**Observaciones.**

- **FIR K = 7:**
  - con bx = 8 y bh = 8, 69,3 dB;
  - 60 dB se alcanzan con bh ≥ 5 y bx ≥ 8 (60,1 dB con bx = 8 y bh = 5);
  - con bh = 4 la SQNR queda entre 50,3 y 50,8 dB, sea cual sea bx; con bx = 6, como máximo 61,1 dB (bh = 8).
- **FIR K = 9:** con bh = 5 da entre 57,8 y 60,0 dB (sin superar 60); con bh ≥ 6 y bx ≥ 8, 61,7 dB o más; con bx = 8 y bh = 8, 69,9 dB.
- **Primer orden:**
  - K<sub>s</sub> = 5: 72,8 dB con bx = bh = 8 y hasta 79,2 dB con bx = 12;
  - K<sub>s</sub> = 7: 70,3 dB con bx = bh = 8;
  - con bh = 5 y bh = 6, K<sub>s</sub> = 5 da lo mismo (63,2 dB con bx = 8), porque los coeficientes redondeados salen idénticos: 0,00195, −0,00977 y 0,01758.

In [ ]:
fig, axs = plt.subplots(1, 4, figsize=(16, 4.2), sharey=True)
vmin = min(T4.values()); vmax = max(T4.values())
for ax, name in zip(axs, cfgs):
    M = np.array([[T4[(name, bx, bh)] for bh in BH] for bx in BX])
    im = ax.imshow(M, origin="lower", cmap="Blues", vmin=vmin, vmax=vmax, aspect="auto")
    for i in range(len(BX)):
        for j in range(len(BH)):
            ax.text(j, i, f"{M[i, j]:.1f}", ha="center", va="center", fontsize=10,
                    color="white" if M[i, j] > (vmin + vmax) / 2 else FIG["ink"],
                    fontweight="bold" if M[i, j] >= 60 else "normal")
    for i in range(len(BX)):
        for j in range(len(BH)):
            if M[i, j] >= 60:
                ax.add_patch(plt.Rectangle((j - 0.5, i - 0.5), 1, 1, fill=False, ec=FIG["orange"], lw=2))
    ax.set_xticks(range(len(BH)), BH); ax.set_yticks(range(len(BX)), BX)
    ax.set_xlabel("bh (bits de coeficientes)"); ax.set_title(name, fontsize=11); ax.grid(False)
axs[0].set_ylabel("bx (bits de x en la corrección)")
fig.colorbar(im, ax=axs, label="SQNR [dB]", shrink=0.85)
fig.suptitle("G3 · SQNR a z = 0,1 km (borde naranja y negrita: ≥ 60 dB)", fontweight="bold")
u.save_fig(fig, "G3_sqnr_bx_bh"); plt.show()

**Observaciones.**

- Todas las configuraciones llegan a 60 dB con bh ≥ 6 y bx ≥ 8. Ninguna llega con bh = 4.
- Con bx = 6, solo bh = 8 supera 60 dB (61,1 a 61,8 dB). La SQNR deja de crecer a partir de bx = 10: la limita bh.

## Parte 3 · ISI a 1,2 sps

Perfiles: gaussiano; Bessel y Butterworth con m = 2, 4 y 6. B = 0,3; 0,4; 0,5; 0,6·R<sub>s</sub> (a 1,2 sps, Nyquist es 0,6·R<sub>s</sub> = 36 GHz y el borde de la señal (1 + 0,1)·R<sub>s</sub>/2 = 0,55·R<sub>s</sub> = 33 GHz).

**Referencia:** el prototipo sin truncar, diseñado con las funciones del golden con L = 8192: `gaussian(B, fs, K=8192, L=8192)` y `analog(kind, B, fs, m, K=8192, d=2048, L=8192)` (pre-retardo de 2048 muestras), con Σ|h|² = 1, aplicado a X. La salida se alinea con la ventana de 64 coeficientes de más energía (o<sub>64</sub>) y las posiciones candidatas de los filtros cortos van de o<sub>64</sub> − 64 a o<sub>64</sub> + 127.

**Criterios** (NRMSE siempre sobre la segunda mitad de la señal, sin 512 muestras por borde de cada mitad):
- **(a)** energía capturada ≥ 98,5 % por K coeficientes contiguos en la mejor posición (como `energy_captured`).
- **(b)** truncar el h largo en esa misma ventana y ajustar solo una ganancia real (sobre la primera mitad).
- **(c)** mínimos cuadrados en frecuencia, peso uniforme solo en |f| ≤ 0,55·R<sub>s</sub> sobre la grilla de 8192 puntos, coeficientes reales, sin usar los datos; retardo entero que minimiza el error en frecuencia.
- **(d)** mínimos cuadrados sobre la señal (primera mitad), coeficientes reales; retardo entero que minimiza el error de ajuste.

K de 1 a 64; si no alcanza, "> 64". K_regla del golden (`K_rule`, definida a 2 sps; para el gaussiano, con m = 4 como el bloque `ISI`) y su equivalente en tiempo a 1,2 sps: 0,6 × K_regla.

In [ ]:
t = time.perf_counter()
TR3, TE3 = u.split_indices(N, 512)
GRAM = u.LagGram(X, TR3)
RES = {}
W_RC = u.rc_weight(GRAM.f_band)  # |H_RRC(f)|² en la banda (D1.2, criterio e)
G_RC = GRAM.G_weighted(W_RC)
M4 = lambda kind, m, B: m in (None, 4) and B <= 0.5 + 1e-9  # casos del criterio (e)
for kind, m in u.PROFILES:
    for B in u.B_RELS:
        wts = {"lsw": (G_RC, W_RC)} if M4(kind, m, B) else None
        RES[(kind, m, B)] = u.isi_case(X, kind, m, B, TR3, TE3, GRAM, weights=wts)
TIMES["parte3_calculo_s"] = time.perf_counter() - t
print(f"{len(RES)} casos en {TIMES['parte3_calculo_s']:.0f} s")

In [ ]:
fmtK = lambda k: ">64" if k is None else k
t5, t5_curve = [], []
CRIT = {"trunc": "b", "lsf": "c", "lsd": "d", "lsw": "e"}
for (kind, m, B), r in RES.items():
    kr = K_rule(B * u.RS, u.RS, m if m else 4)
    row = {"perfil": u.profile_name(kind, m), "B_Rs": B, "K_regla": kr, "0,6·K_regla": 0.6 * kr,
           "a (98,5 %)": fmtK(u.min_K(r["energy"], lambda e: e >= u.ENERGY_MIN))}
    for key, lab in CRIT.items():
        for tol in u.TOLS:
            row[f"{lab} {tol:g}"] = fmtK(u.min_K(r[key], lambda e, tol=tol: e <= tol)) if key in r else "—"
    t5.append(row)
    for K in r["energy"]:
        t5_curve.append({"perfil": u.profile_name(kind, m), "B_Rs": B, "K": K, "energia": r["energy"][K],
                         "nrmse_b": r["trunc"][K], "nrmse_c": r["lsf"][K], "nrmse_d": r["lsd"][K],
                         "nrmse_e": r["lsw"][K] if "lsw" in r else None})
show(t5)
u.save_csv(t5, "T5_isi_K_minimo"); u.save_csv(t5_curve, "T5_isi_curvas_vs_K")

**Observaciones.**

- **(a) energía ≥ 98,5 %:** K ≤ 16 en los 28 casos (máximo: Bessel y Butterworth m = 2 con B = 0,6). No garantiza el NRMSE: Bessel m = 4 con B = 0,6 cumple (a) con K = 13, pero (b) necesita 51 para 1e-2.
- **(b) truncado:** muy disperso. Con 2e-3 da "> 64" en 17 de los 24 casos analógicos. Con 1e-2 va de 7 a 58 (58: Bessel m = 2 y Butterworth m = 2, B = 0,6).
- **(c) y (d), mínimos cuadrados:**
  - máximo sobre los 28 casos: (c) 18 / 21 / 30 y (d) 14 / 17 / 26 para 1e-2 / 6e-3 / 2e-3 (Butterworth m = 2, B = 0,6);
  - el gaussiano necesita K ≤ 9 en los dos;
  - comparado con 0,6·K_regla (9,6 o 19,2), (d) a 1e-2 queda por encima de 9,6 en 11 de los 18 casos con K_regla = 16.

**Por qué (b) no es monótono en B.** El prototipo se diseña muestreando H<sub>P</sub>(f) en la grilla de la FFT (`analog`). Como h es real, en f<sub>s</sub>/2 el espectro periódico salta de H<sub>P</sub>(f<sub>s</sub>/2) a su conjugado: un salto de 2·|H<sub>P</sub>(f<sub>s</sub>/2)|·|sen φ(f<sub>s</sub>/2)| (el pre-retardo de 2048 muestras no cambia la fase en Nyquist). Ese salto produce una cola que decae como 1/n y que el truncado no puede representar. La celda siguiente lo mide al lado del K de (b).

In [ ]:
from scipy import signal as sps
from emulador.isi.profiles import analog_prototype
nyq = []
for kind, m in u.PROFILES[1:]:
    for B in u.B_RELS:
        b_, a_ = analog_prototype(kind, B * u.RS, m)
        _, Hn = sps.freqs(b_, a_, worN=[2 * np.pi * u.FS / 2])
        row5 = next(r for r in t5 if r["perfil"] == u.profile_name(kind, m) and r["B_Rs"] == B)
        nyq.append({"perfil": u.profile_name(kind, m), "B_Rs": B, "|H(fs/2)|": abs(Hn[0]),
                    "|sen fase|": abs(np.sin(np.angle(Hn[0]))), "salto 2|H||sen|": 2 * abs(Hn[0].imag),
                    "K (b) 1e-2": row5["b 0.01"], "K (d) 1e-2": row5["d 0.01"]})
show(nyq, d=3)
u.save_csv(nyq, "T5_salto_en_nyquist")

**Observaciones.**

- Los 6 casos con salto ≤ 0,122 necesitan entre 7 y 12 coeficientes con (b) y 1e-2. Por ejemplo: Bessel m = 4 con B = 0,4 (salto 0,049, K = 7), Bessel m = 6 con B = 0,5 (0,121, K = 7) y Butterworth m = 4 con B = 0,6 (salto 0, K = 9).
- Los de mayor salto (Butterworth m = 2 y m = 6 con B = 0,6: 1,41; Bessel m = 2 con B = 0,6: 1,36) necesitan 58, 56 y 58 con (b). Con (d) bajan a 14: los mínimos cuadrados solo miran la banda de la señal (≤ 0,55·R<sub>s</sub>), y el salto está en 0,6·R<sub>s</sub>.

In [ ]:
num = lambda k: 68 if k == ">64" else k
t6 = []
m4 = {u.profile_name(k, m) for k, m in u.PROFILES if m in (None, 4)}
for Bmax in (0.4, 0.5):
    sel = [r for r in t5 if r["perfil"] in m4 and r["B_Rs"] <= Bmax + 1e-9]
    row = {"B <=": f"{Bmax}·Rs"}
    for col in [c for c in t5[0] if c[0] in "abcde" and c != "B_Rs"]:
        worst = max(sel, key=lambda r: num(r[col]))
        row[col] = f"{worst[col]} ({worst['perfil']}, {worst['B_Rs']})"
    t6.append(row)
show(t6)
u.save_csv(t6, "T6_isi_peor_caso_m4")

**Observaciones.**

- **B ≤ 0,4·R<sub>s</sub>** (gaussiano, Bessel m = 4, Butterworth m = 4): (c) pide 11 / 13 / 19 y (d) 9 / 11 / 15 para 1e-2 / 6e-3 / 2e-3. (b) pide 15 / 27 / > 64.
- **B ≤ 0,5·R<sub>s</sub>:** (c) pide 13 / 17 / 24 y (d) 11 / 13 / 20, siempre por Bessel m = 4 con B = 0,5. (b) pide 30 / 51 / > 64.
- (a) pide 4 (B ≤ 0,4) y 5 (B ≤ 0,5).

**Observaciones.**

- **(e), mínimos cuadrados en frecuencia pesados por |H<sub>RRC</sub>(f)|², sin datos,** da el **mismo K mínimo que (d)** en los 9 casos (3 perfiles con m = 4, B = 0,3 a 0,5): peor caso 9 / 11 / 15 con B ≤ 0,4 y 11 / 13 / 20 con B ≤ 0,5 para 1e-2 / 6e-3 / 2e-3. (c), con peso uniforme, pide 11 / 13 / 19 y 13 / 17 / 24.
- Con K fijo el NRMSE de (e) coincide con el de (d) en 4 cifras: Bessel m = 4, B = 0,5, K = 16 da 3,38633e-3 (e) contra 3,38607e-3 (d) y 6,14e-3 (c). El espectro de la señal es |H<sub>RRC</sub>|², así que pesar por él equivale a ajustar sobre la señal sin usar los datos.

### T5b · Acumulador de la ISI con el h elegido (D1.2)

Para los diseños (c) y (e) con K = 12 y K = 16, los 3 perfiles con m = 4 y B ≤ 0,5·R<sub>s</sub>, con h tal como sale del diseño (aproxima al prototipo con Σ|h|² = 1, sin reescalar):
- **cota** = Σ|h| × pico de entrada (0,698);
- **pico real** sobre la señal común (4 canales, sin 1024 muestras por borde): de la salida y de las sumas parciales de un FIR en forma directa, acumulando en el orden de los coeficientes (k = 0 … K−1);
- **bits enteros** I (con signo, `ac_fixed<W, I>`) para cada valor: el menor I con 2<sup>I−1</sup> > valor.

In [ ]:
t5b = []
for kind, m in [(k, mm) for k, mm in u.PROFILES if mm in (None, 4)]:
    for B in (0.3, 0.4, 0.5):
        r = RES[(kind, m, B)]
        for key, lab in (("lsf", "c"), ("lsw", "e")):
            for K in (12, 16):
                h, s0 = r["h"][K][key]
                bound = float(np.sum(np.abs(h))) * PEAK_IN
                out_pk, part_pk = u.accumulator_peaks(CH, h, inner)
                t5b.append({"perfil": u.profile_name(kind, m), "B_Rs": B, "diseño": lab, "K": K,
                            "NRMSE": r[key][K], "Σ|h|": float(np.sum(np.abs(h))), "cota": bound,
                            "pico salida": out_pk, "pico parciales": part_pk,
                            "I cota": u.int_bits(bound), "I salida": u.int_bits(out_pk), "I parciales": u.int_bits(part_pk)})
show(t5b, d=4)
u.save_csv(t5b, "T5b_isi_acumulador")

**Observaciones.**

- **Cota Σ|h|·pico de entrada:** entre 0,860 y 1,319. Pide **2 bits enteros en 24 de los 36 casos** (todos los Butterworth m = 4, Bessel m = 4 con B = 0,3 y 0,5, y gaussiano con B = 0,3) y 1 bit en los otros 12.
- **Pico real sobre la señal común:**
  - la salida llega como máximo a 0,7208 (Butterworth m = 4, B = 0,5, (e), K = 12);
  - las sumas parciales en forma directa, a 0,7312, hasta 0,026 por encima de la salida del mismo caso;
  - **1 bit entero alcanza en los 36 casos.**
- La diferencia entre la cota y el pico real (1,319 contra 0,637 en Butterworth m = 4, B = 0,3, (c), K = 16) es el margen entre el peor caso de cualquier señal con pico 0,698 y esta señal en particular. El pico de las sumas parciales depende además del orden de acumulación (acá, k = 0 … K−1).

In [ ]:
def plot_K(ax, col, title):
    for kind, m in u.PROFILES:
        name = u.profile_name(kind, m)
        ks = [num(r[col]) for r in t5 if r["perfil"] == name]
        ax.plot(u.B_RELS, ks, color=FAMILY[kind], marker=MARK[m], ms=7, lw=1.6,
                ls="-" if m in (None, 4) else (":" if m == 2 else "--"), label=name,
                markerfacecolor="white" if m == 2 else FAMILY[kind])
    ax.axhline(66, color=FIG["muted"], lw=0.8)
    ax.set_ylim(0, 71); ax.set_yticks([0, 16, 32, 48, 64, 68], ["0", "16", "32", "48", "64", ">64"])
    ax.set_xticks(u.B_RELS); ax.set_title(title, fontsize=10)

fig, axs = plt.subplots(3, 4, figsize=(17, 11), sharex=True, sharey=True)
for i, tol in enumerate(u.TOLS):
    plot_K(axs[i, 0], "a (98,5 %)", "(a) energía ≥ 98,5 %" if i == 0 else "(a) energía ≥ 98,5 % (igual)")
    for j, lab in enumerate(("b", "c", "d")):
        name = {"b": "truncado + ganancia", "c": "MC en frecuencia", "d": "MC sobre la señal"}[lab]
        plot_K(axs[i, j + 1], f"{lab} {tol:g}", f"({lab}) {name}, NRMSE ≤ {tol:g}")
for ax in axs[-1]:
    ax.set_xlabel("B / Rs")
for ax in axs[:, 0]:
    ax.set_ylabel("K mínimo")
h, l = axs[0, 0].get_legend_handles_labels()
fig.legend(h, l, loc="outside lower center", ncol=7, fontsize=10)
fig.suptitle("G4 · Menor K contra B por perfil, un panel por criterio y tolerancia (> 64: no alcanza)", fontweight="bold")
u.save_fig(fig, "G4_isi_K_vs_B"); plt.show()

**Observaciones.**

- En los paneles (c) y (d) las curvas son suaves y crecen con B. En (b) saltan, por el efecto de Nyquist de la tabla anterior.
- Con (d) y 2e-3, el peor caso es 26 (Butterworth m = 2, B = 0,6); el gaussiano queda en 7 a 9.

In [ ]:
K_G5 = 16
Lp = u.L_REF
fs_ = u.FS
fig, axs = plt.subplots(2, 2, figsize=(13, 7.5), sharex=True)
for j, B in enumerate((0.5, 0.6)):
    r = RES[("bessel", 4, B)]
    o64 = r["o64"]
    curves = {"prototipo (8192)": (r["h_ref"], -o64, FIG["ink"], "-"),
              f"truncado + ganancia, K = {K_G5}": (*r["h"][K_G5]["trunc"], FIG["orange"], "--"),
              f"MC en frecuencia, K = {K_G5}": (*r["h"][K_G5]["lsf"], FIG["aqua"], "-")}
    g0 = abs(np.sum(r["h_ref"]))  # |H_ref(0)|: el módulo se grafica relativo a él
    for lab, (h, s, col, ls) in curves.items():
        H = np.fft.fft(h, Lp)
        f, tau = group_delay(H, fs_)
        mag = np.fft.fftshift(20 * np.log10(np.abs(H) / g0 + 1e-12))
        keep = (f >= 0) & (f <= 0.6 * u.RS)
        axs[0, j].plot(f[keep] / u.RS, mag[keep], color=col, ls=ls, lw=1.8, label=lab)
        axs[1, j].plot(f[keep] / u.RS, tau[keep] * fs_ + s, color=col, ls=ls, lw=1.8, label=lab)
    for ax in axs[:, j]:
        for fx, txt in ((0.55, "borde de la señal 0,55·Rs"), (0.6, "Nyquist 0,6·Rs")):
            ax.axvline(fx, color=FIG["muted"], ls=":", lw=1.2)
        ax.set_xlim(0, 0.61)
    axs[0, j].text(0.548, 0.02, "borde señal", rotation=90, transform=axs[0, j].get_xaxis_transform(),
                   color=FIG["muted"], fontsize=9, ha="right")
    axs[0, j].text(0.598, 0.02, "Nyquist", rotation=90, transform=axs[0, j].get_xaxis_transform(),
                   color=FIG["muted"], fontsize=9, ha="right")
    axs[0, j].set_title(f"Bessel m = 4, B = {B}·Rs", fontsize=11)
    axs[0, j].set_ylim(-10, 2)
    axs[1, j].set_xlabel("f / Rs")
axs[0, 0].set_ylabel("|H| / |H_ref(0)| [dB]"); axs[1, 0].set_ylabel("τ [muestras desde o64]")
axs[0, 0].legend(fontsize=9, loc="lower left")
fig.suptitle(f"G5 · Módulo y retardo de grupo: prototipo, truncado y mínimos cuadrados con K = {K_G5}", fontweight="bold")
u.save_fig(fig, "G5_bessel4_modulo_retardo"); plt.show()
for B in (0.5, 0.6):
    r = RES[("bessel", 4, B)]
    print(f"B = {B}: NRMSE con K = {K_G5}: truncado {r['trunc'][K_G5]:.2e}, MC frecuencia {r['lsf'][K_G5]:.2e}, MC señal {r['lsd'][K_G5]:.2e}")
TIMES["parte3_s"] = time.perf_counter() - t

**Observaciones.**

- Con K = 16, Bessel m = 4: a B = 0,5, NRMSE 1,82e-2 (truncado), 6,14e-3 (MC en frecuencia) y 3,39e-3 (MC sobre la señal); a B = 0,6, 3,10e-2, 1,05e-2 y 5,78e-3.
- En la figura, los dos filtros de 16 coeficientes se separan del prototipo sobre todo entre el borde de la señal (0,55·R<sub>s</sub>) y Nyquist (0,6·R<sub>s</sub>): el módulo cae y el retardo de grupo sube. El truncado ondula además dentro de la banda, en módulo y en retardo; el de mínimos cuadrados en frecuencia, menos.

## Parte 4 · DSP por fórmula (estimación por fórmula, no síntesis)

- **ISI, 4 canales reales, con FFA:** para cada (a, b) con 2<sup>a</sup> | P y 3<sup>b</sup> | P, K<sub>p</sub> es el menor múltiplo de 2<sup>a</sup>·3<sup>b</sup> que sea ≥ K y el costo es K<sub>p</sub>·(3/4)<sup>a</sup>·(2/3)<sup>b</sup> multiplicaciones por muestra; se toma el mínimo. DSP = 4·P·costo.
- **CD de primer orden (X e Y), 2 productos de 8 bits por DSP:** DSP = 2·P·K<sub>u</sub>, con K<sub>u</sub> = (K<sub>s</sub> + 1)/2.
- Porcentaje sobre 4272 DSP.

In [ ]:
t = time.perf_counter()
PS = (48, 96, 192)
t7_isi, t7_cd = [], []
for K in (8, 12, 16, 24, 32):
    for P in PS:
        c = u.ffa_cost(K, P)
        dsp = 4 * P * c["mult_per_sample"]
        t7_isi.append({"K": K, "P": P, "a": c["a"], "b": c["b"], "K_p": c["K_p"],
                       "mult/muestra": c["mult_per_sample"], "DSP": dsp, "% de 4272": 100 * dsp / u.DSP_TOTAL})
for Ks in (3, 5, 7, 9):
    for P in PS:
        Ku = (Ks + 1) // 2
        dsp = 2 * P * Ku
        t7_cd.append({"K_s": Ks, "K_u": Ku, "P": P, "DSP": dsp, "% de 4272": 100 * dsp / u.DSP_TOTAL})
display(Markdown("**T7 · ISI con FFA (estimación por fórmula)**")); show(t7_isi, d=4)
display(Markdown("**T7 · CD de primer orden (estimación por fórmula)**")); show(t7_cd, d=4)
u.save_csv(t7_isi, "T7_dsp_isi_ffa"); u.save_csv(t7_cd, "T7_dsp_cd_primer_orden")

**Observaciones.**

- **ISI con FFA:**
  - K = 16 cuesta 5,06 mult/muestra (a = 4, b = 0): 972, 1944 y 3888 DSP con P = 48, 96 y 192 (22,8 %, 45,5 % y 91,0 % de 4272);
  - con P = 192, K = 24 (5184, 121 %) y K = 32 (5832, 137 %) superan el total;
  - con K = 32 y P = 48 solo se puede a = 4 (48 = 2⁴·3) y cuesta 10,1 mult/muestra; con P = 96 se puede a = 5 y baja a 7,59.
- **CD de primer orden:** K<sub>s</sub> = 5 cuesta 288, 576 y 1152 DSP (6,7 %, 13,5 % y 27,0 %); K<sub>s</sub> = 7, 384, 768 y 1536 (9,0 %, 18,0 % y 36,0 %).
- Con f<sub>s</sub> = 72 GSa/s, P = 48, 96 y 192 corresponden a f<sub>clk</sub> = 1,5 GHz, 750 MHz y 375 MHz.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(13, 4.6), sharey=True)
ramp5 = ["#86b6ef", "#5598e7", "#2a78d6", "#1c5cab", "#0d366b"]
for col, K in zip(ramp5, (8, 12, 16, 24, 32)):
    axs[0].plot(PS, [r["DSP"] for r in t7_isi if r["K"] == K], color=col, marker="o", ms=7, label=f"K = {K}")
ramp4 = ["#f4a582", "#eb6834", "#c4501f", "#8f3510"]
for col, Ks in zip(ramp4, (3, 5, 7, 9)):
    axs[1].plot(PS, [r["DSP"] for r in t7_cd if r["K_s"] == Ks], color=col, marker="s", ms=7, label=f"K_s = {Ks}")
for ax, title in zip(axs, ("ISI, 4 canales, FFA", "CD de primer orden, X e Y")):
    ax.axhline(u.DSP_TOTAL, color=FIG["muted"], ls="--", lw=1.2)
    ax.text(PS[0], u.DSP_TOTAL, " 4272 DSP", color=FIG["muted"], fontsize=9, va="bottom")
    ax.set_xticks(PS); ax.set_xlabel("P (muestras por ciclo por canal)"); ax.set_title(title, fontsize=11)
    ax.legend(fontsize=9)
axs[0].set_ylabel("DSP")
fig.suptitle("G6 · DSP contra paralelismo (estimación por fórmula)", fontweight="bold")
u.save_fig(fig, "G6_dsp_vs_P"); plt.show()
TIMES["parte4_s"] = time.perf_counter() - t

**Observaciones.**

- Solo la ISI con K ≥ 24 y P = 192 supera la línea de 4272. La CD de primer orden queda por debajo de 1920 DSP (K<sub>s</sub> = 9, P = 192).

## Metadatos y tiempo de corrida

In [ ]:
TIMES["total_s"] = time.perf_counter() - T0
meta = {
    "notebook": "08_escenario_60gbd_1p2sps.ipynb", "tarea": "D1.1",
    "fecha_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "git": u.git_info(),
    "versiones": {"numpy": np.__version__, "scipy": scipy.__version__},
    "escenario": u.ESC.to_dict(),
    "senal_comun": {
        "n_simbolos": u.N_SYM, "n_muestras_por_canal": int(N), "modulacion": "DP-16QAM (PAM4 Gray por tributario)",
        "gray": {"00": -3, "01": -1, "11": 1, "10": 3}, "escala_niveles": "1/3",
        "prbs": {"orden": u.PRBS_ORDER, "polinomio": "x^15 + x^14 + 1", "invertida": False,
                 "semilla_base": u.BASE_SEED, "periodo": u.PERIOD, "bits_por_simbolo": 2, "orden_bits": "MSB primero"},
        "canales": {c: {"offset_bits": o, "semilla": s, "rms": float(r)}
                    for c, o, s, r in zip(u.CHANNELS, u.OFFSETS, SIG["seeds"], SIG["rms"])},
        "rrc": {"rolloff": u.ESC.rolloff, "span_transmisor": u.SPAN_TX, "span_filtro_adaptado_rx": u.SPAN_RX,
                "conformado_sps": u.ESC.sps_num, "diezmado": u.ESC.sps_den},
        "rms_objetivo": u.RMS_TARGET, "factor_comun": SIG["scale"],
    },
    "formato": {"q": "Q1.15 = ac_fixed<16, 1>", "redondeo": "half_even", "desborde": "saturacion"},
    "cd": {"D_ps_nm_km": u.D_PS, "lambda_m": u.LAM, "S_ps_nm2_km": list(u.S_PS),
           "z_T1_km": list(Z_T1), "z_T2_km": list(Z_T2), "z_T3_km": list(Z_T3), "K_T2": "impar 1..41",
           "ajuste": "primera mitad, medicion segunda mitad, 256 muestras por borde"},
    "t4": {"z_km": 0.1, "S": 0.0, "bx": list(BX), "bh": list(BH), "configs": list(cfgs)},
    "isi": {"perfiles": [u.profile_name(k, m) for k, m in u.PROFILES], "B_Rs": list(u.B_RELS),
            "L_ref": u.L_REF, "d_ref": u.D_REF, "K_max": u.K_MAX_SEARCH, "tolerancias": list(u.TOLS),
            "energia_min": u.ENERGY_MIN, "banda_ls_Hz": u.BAND_EDGE, "lags_relativos": [u.LAG_MIN, u.LAG_MAX - 1],
            "ajuste": "primera mitad, medicion segunda mitad, 512 muestras por borde", "K_G5": K_G5},
    "d1_2": {"criterio_e": "MC en frecuencia pesado por |H_RRC(f)|^2 (roll-off 0,1), sin datos; m = 4 y B <= 0,5 Rs",
             "acumulador": "forma directa, sumas parciales en el orden k = 0..K-1; disenos c y e, K = 12 y 16",
             "z_T1b_km": list(Z_T1B)},
    "t7": {"dsp_total": u.DSP_TOTAL, "P": list(PS), "nota": "estimacion por formula, no sintesis"},
    "tiempos_s": {k: round(v, 1) for k, v in TIMES.items()},
}
u.save_meta(meta)
print({k: round(v, 1) for k, v in TIMES.items()})
print("guardado en", u.OUT)

## Para decidir

Los datos están arriba; esta sección solo los junta, **sin decidir**.

**Ancho de banda B de la ISI**
- Con mínimos cuadrados, el peor caso crece con B: con (d) y 1e-2, K va de 5 a 12 con B = 0,3 y de 4 a 14 con B = 0,6, según el perfil (T5).
- Con truncado, el K depende sobre todo del salto del prototipo en Nyquist, no de B (tabla de Nyquist).
- A 1,2 sps, B = 0,6·R<sub>s</sub> coincide con Nyquist: el −3 dB del prototipo cae justo en f<sub>s</sub>/2.

**K_max de la ISI**
- Peor caso sobre los 7 perfiles: (d) 14 / 17 / 26 y (c) 18 / 21 / 30 para 1e-2 / 6e-3 / 2e-3.
- Solo m = 4 con B ≤ 0,5: (d) 11 / 13 / 20 y (c) 13 / 17 / 24 (T6).
- K_MAX actual del golden: 32 (decisión 6); equivale a 19,2 a 1,2 sps.

**Diseño y acumulador de la ISI (D1.2)**
- (e), MC pesado por |H<sub>RRC</sub>|², da los mismos K que (d) sin usar datos; (c) pide 2 a 4 coeficientes más en el peor caso con m = 4 (T5, T6).
- Acumulador con K = 12 y 16 (m = 4, B ≤ 0,5): la cota pide hasta 2 bits enteros (máximo 1,319) y el pico real sobre la señal común, 1 bit (máximo 0,7312) (T5b).

**Tolerancia de la ISI**
- Pasar de 1e-2 a 2e-3 cuesta entre 3 y 13 coeficientes más con (c) y entre 2 y 12 con (d), según el caso (T5).
- Costo en DSP con FFA a P = 96: K = 12 → 1728, K = 16 → 1944, K = 24 → 2592, K = 32 → 2916 (T7).

**Arquitectura de la CD a 100 m**
- Rango: sin saturación hasta 10 km (pico 0,9965); desde 12 km satura con 1 bit entero (2 muestras, 57,6 dB; T1b).
- **FIR complejo:** K = 5 (7,29e-4) o K = 7 (3,14e-4). Con 3e-4 hace falta K = 9 (T2).
- **Primer orden:** K<sub>s</sub> = 5 (7,71e-4) o K<sub>s</sub> = 7 (4,05e-4). Se estanca por encima de 1e-3 desde 0,2 km; la extendida con K<sub>c</sub> = 3 lo mejora sin alcanzar al FIR (T3).
- **Precisión:** ≥ 60 dB con bx ≥ 8 y bh ≥ 6 en las cuatro configuraciones; 69,3 a 72,8 dB con bx = bh = 8 (T4).
- **DSP:** primer orden K<sub>s</sub> = 5 o 7 con P = 96: 576 o 768 (T7). El FIR complejo no está en T7.

**Pendientes que no resuelve este notebook**
- Decisiones 15 y 32 en revisión por el documento común.
- La decisión 34 (pico 0,95 del fondo de escala de `ac_fixed<14, 1>`) no coincide con RMS 0,25 en Q1.15.
- Las 4 PRBS15 están correlacionadas a desfasajes de más de 12 000 símbolos: con un solo polinomio son desplazamientos de la misma secuencia m. **Pendiente con el equipo** (polinomio distinto por tributario o PRBS más larga).